In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as si
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")
import QuantLib as ql
import plotly.graph_objects as go
import os
import regex as re
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy import interpolate
from scipy.optimize import minimize
import seaborn as sns
from tqdm import tqdm
from datetime import datetime, timedelta

In [ ]:
#这块是提取要用的数据
data_path = "price_data_minus2_to_2.csv"

tmp = pd.read_csv(data_path)

# 确保 QUOTE_DATE 列为 datetime64 类型
#tmp['QUOTE_DATE'] = pd.to_datetime(tmp['QUOTE_DATE'])

# 筛选 2023 2022 2021 2020 2019 2018 2017 2016 2015 2014年的数据
#tmp = tmp[tmp['QUOTE_DATE'].dt.year == 2023]
#tmp = tmp[tmp['DTE'] == 365 * 2].reset_index(drop=True)

tmp

In [ ]:
#sort dates
dates = sorted(["2023-04-19",
"2023-02-24",
"2023-06-06",
"2023-03-15",
"2023-07-05",
"2023-05-12",
"2023-01-26",
"2023-02-02",
"2023-01-11",
"2023-08-11",
"2023-02-16",
"2023-04-04",
"2023-08-02",
"2023-09-15",
"2023-06-02",
"2023-02-08",
"2023-02-28",
"2023-03-23",
"2023-04-03",
"2023-07-14"])
# Filter the dataframe for these dates
tmp = tmp[tmp['date'].isin(dates)]
tmp

In [ ]:
#getting ssvi iv
def compute_theta_t(params, t, kappa1, kappa2):
    V, V_prime, theta = params
    theta_t = theta * t + (V - theta) * (1 - np.exp(-kappa1 * t)) / kappa1 + \
              (V_prime - theta) * kappa1 * ((1 - np.exp(-kappa2 * t)) / kappa2 - \
              (1 - np.exp(-kappa1 * t)) / kappa1) / (kappa1 - kappa2)
    return theta_t

def compute_phi(params_w_t, params_theta_t, t, phi_type, kappa1, kappa2):
    rho, eta, gamma = params_w_t
    theta_t = compute_theta_t(params_theta_t, t, kappa1, kappa2)
    if phi_type == "POWER-LAW1":
        return eta / theta_t ** gamma
    elif phi_type == "POWER-LAW2":
        return eta / (theta_t ** gamma * (1 + theta_t) ** (1 - gamma))
    elif phi_type == "SQRT":
        return eta / np.sqrt(theta_t)
    elif phi_type == "HESTON":
        return (1 - (1 - np.exp(-gamma * theta_t)) / (gamma * theta_t)) / (gamma * theta_t)
    else:
        raise ValueError("Invalid phi_type")

def compute_ssvi_inference(params_w_t, params_theta_t, t, k, phi_type, kappa1, kappa2):
    rho, eta, gamma = params_w_t
    theta_t = compute_theta_t(params_theta_t, t, kappa1, kappa2)
    phi = compute_phi(params_w_t, params_theta_t, t, phi_type, kappa1, kappa2)
    return 0.5 * theta_t * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + 1 - rho ** 2))


# 定义 kappa1 和 kappa2
kappa1 = 5.5
kappa2 = 0.1

# 使用 apply 方法，传递所有必要参数
tmp['ssvi_iv'] = tmp.apply(
    lambda row: np.sqrt(
        compute_ssvi_inference(
            params_w_t=[row['rho'], row['eta'], row['gamma']],
            params_theta_t=[row['V'], row['V_prime'], row['theta']],
            t=2,#now it's 2 years
            k=row['Log-Strike'],
            phi_type="POWER-LAW1",
            kappa1=kappa1,
            kappa2=kappa2
        ) / 2
    ), axis=1
)

tmp

In [ ]:
tmp.reset_index(drop=True, inplace=True)
tmp

In [ ]:
#计算binomial美式价格
def binomial_tree_american_option(S0, K, T, r, sigma, N, q, option_type="call"):
    """
    Prices an American option using the binomial tree model with dividends.
    
    Parameters:
    S0 : float - Initial stock price
    K : float - Strike price
    T : float - Time to maturity in years
    r : float - Risk-free interest rate
    sigma : float - Volatility
    N : int - Number of steps in the binomial tree
    q : float - Dividend yield
    option_type : str - Type of the option ("call" or "put")
    
    Returns:
    float - Estimated option price
    """  
    dt = T / N
    u = np.exp((r - q) * dt + sigma * np.sqrt(dt))
    d = np.exp((r - q) * dt - sigma * np.sqrt(dt))
    p = (np.exp((r - q) * dt) - d) / (u - d)

    # 初始化股票价格树
    stock_prices = np.asarray([S0 * u**j * d**(N-j) for j in range(N+1)])
    
    # 初始化期权价值树的最终节点
    if option_type == "call":
        option_values = np.maximum(stock_prices - K, 0)
    else:  # put
        option_values = np.maximum(K - stock_prices, 0)

    # 反向迭代期权价值树
    for i in range(N-1, -1, -1):
        stock_prices = S0 * u**np.arange(i+1) * d**(i-np.arange(i+1))
        option_values_early = np.maximum(stock_prices - K, 0) if option_type == "call" else np.maximum(K - stock_prices, 0)
        option_values = np.exp(-r * dt) * (p * option_values[1:] + (1 - p) * option_values[:-1])
        option_values = np.maximum(option_values, option_values_early)  # 早期行权的检查

    return option_values[0]

# 设置二叉树的步数
#N = 100  # 或者根据需要调整

############################################################################################################

def bisect(f, a, b, tol):
    assert a < b and f(a) * f(b) < 0, "The assumptions of the bisection method are not met."
    c = (a + b) / 2
    while (b - a) / 2 > tol:
        c = (a + b) / 2
        if abs(f(c)) < tol:
            return c
        else:
            if f(a) * f(c) < 0:
                b = c
            else:
                a = c
    return c

# def bisect(f, a, b, tol):
#     if f(a) * f(b) >= 0:
#         return None  # 如果a和b之间不存在根，则返回None
#     c = (a + b) / 2
#     while (b - a) / 2 > tol:
#         if abs(f(c)) < tol:
#             return c
#         elif f(a) * f(c) < 0:
#             b = c
#         else:
#             a = c
#         c = (a + b) / 2
#     return c

############################################################################################################

def amer_implied_vol(S0, r, T, K, q, market_price, option_type="call"):
    """
    Bisection algorithm to find the implied American volatility using the bisect function.
    """
    # 定义函数f，它计算给定波动率下的期权价格与市场价格之差
    f = lambda sigma: binomial_tree_american_option(S0, K, T, r, sigma, N=100, q=q, option_type=option_type) - market_price

    sigma_min = 0.01
    sigma_max = 4.00
    eps = 1e-5

    # 使用bisect函数查找隐含波动率
    sigma_iv = bisect(f, sigma_min, sigma_max, eps)
    
    # 返回找到的隐含波动率
    return sigma_iv

In [ ]:
import pandas as pd


tmp['implied_vol_put'] = pd.NA
tmp['implied_vol_call'] = pd.NA

# for index, row in tmp.iterrows():
#     # 计算看跌期权的隐含波动率
#     implied_vol_put = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_put'], option_type="put")
#     # 计算看涨期权的隐含波动率
#     implied_vol_call = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_call'], option_type="call")
    
#     # 打印每次计算得到的隐含波动率值
#     print(f"Index {index}: Implied Vol Put = {implied_vol_put}, Implied Vol Call = {implied_vol_call}")

#     # 直接在DataFrame中更新隐含波动率值
#     tmp.at[index, 'implied_vol_put'] = implied_vol_put
#     tmp.at[index, 'implied_vol_call'] = implied_vol_call

for index, row in tmp.iterrows():
    try:
        # Calculate implied volatility for put options
        implied_vol_put = amer_implied_vol(row['S'], row['r'], 11, row['strike'], 0, row['amer_put'], option_type="put")
        tmp.at[index, 'implied_vol_put'] = implied_vol_put
    except AssertionError:
        print(f"Skipping row {index} for put option due to bisection method assumptions not being met.")
    
#     try:
#         # Calculate implied volatility for call options
#         implied_vol_call = amer_implied_vol(row['S'], row['r'], 11, row['strike'], 0, row['amer_call'], option_type="call")
#         tmp.at[index, 'implied_vol_call'] = implied_vol_call
#     except AssertionError:
#         print(f"Skipping row {index} for call option due to bisection method assumptions not being met.")

tmp

In [ ]:
#tmp.to_csv('simuvol.csv', index=False)

In [ ]:
#dealing with missing data

In [ ]:
!pip install python-docx

from docx import Document

# Load the .docx file
doc_path = 'row.docx'  # Update this path to your document's location
doc = Document(doc_path)



data_list = []

# Extract data from each paragraph
for paragraph in doc.paragraphs:
    if paragraph.text.strip():  # Ensure the paragraph is not empty
        data_list.append(paragraph.text.strip())  # Add the paragraph text to the list

# Convert the list into a DataFrame
df = pd.DataFrame(data_list, columns=['Data'])

# Display the DataFrame to check the output
print(df)

# # Initialize a list to store the extracted data
# da = []

# # Extract row numbers and the corresponding 'put' (1) or 'call' (0)
# for paragraph in doc.paragraphs:
#     if paragraph.text.strip():  # Ensure the paragraph is not empty
#         parts = paragraph.text.split()
#         number = int(parts[0])  # Convert the first part to an integer
# #         type_value = 1 if parts[1].lower() == 'put' else 0  # Determine type based on the second part
# #         da.append({'Number': number, 'Type': type_value})

# # Convert the list of dictionaries into a DataFrame
# df = pd.DataFrame(da)